**Navigation** : [Index](README.md) | [SW-10 C#](SW-10-RDFStar-CSharp.ipynb) | [SW-10 Python — source du portage](SW-10-RDFStar-Python.ipynb) | [SW-11 Python >>](SW-11-Python-KnowledgeGraphs.ipynb)

# SW-10-RDFStar-Rust

## RDF 1.2 (RDF-Star) : assertions sur des assertions, en Rust

> Ce carnet est le **jumeau Rust** du carnet [SW-10-RDFStar-Python](SW-10-RDFStar-Python.ipynb) (rdflib), lui-meme miroir du [SW-10-RDFStar-CSharp](SW-10-RDFStar-CSharp.ipynb) (dotNetRDF). Meme sujet, memes mesures, autre pile : la famille **oxigraph** (`oxrdf`, `oxttl`, `oxrdfxml`, `oxjsonld`, `oxigraph`), toutes sous licence `MIT OR Apache-2.0`.

### Duree estimee : 45 minutes

A la fin de ce carnet, vous saurez :

1. **Expliquer** pourquoi un triplet RDF ne peut pas porter d'annotation, et ce que la reification ajoute au graphe
2. **Construire** une reification a la main, puis avec une fonction utilitaire, en `oxrdf`
3. **Negocier** la difference entre le modele RDF 1.2 (`Term::Triple`, derriere une feature cargo) et la reification portable
4. **Interroger** les annotations en SPARQL avec `oxigraph::Store` charge en memoire
5. **Comparer** reification, graphes nommes et quoted triples sur des criteres mesures : nombre de triplets, expressivite, interoperabilite

Le texte pedagogique suit celui du jumeau Python ; les exemples executes ici sont en Rust, sur le noyau `rust` (evcxr).

## 0. Installation des dependances

La pile vient de **oxigraph**, un projet Rust dedie a RDF (base de donnees, moteur SPARQL, parsers, serialiseurs). Le carnet charge cinq crates, tous de la meme famille et de la meme licence (`MIT OR Apache-2.0`) :

- **oxrdf** — le modele de donnees : `NamedNode`, `BlankNode`, `Literal`, `Triple`, `Term`, `NamedOrBlankNode`, et le conteneur `Graph` (ensemble de triplets en memoire) ;
- **oxttl** — Turtle et N-Triples : `TurtleSerializer`, `NTriplesSerializer` ;
- **oxrdfxml** — RDF/XML : `RdfXmlSerializer` ;
- **oxjsonld** — JSON-LD : `JsonLdSerializer` ;
- **oxigraph** — le moteur SPARQL (`Store`, `QueryResults`).

Le crate `oxigraph` est charge avec `default-features = false` : ses features par defaut tirent le backend de stockage RocksDB (`oxrocksdb-sys`, C++), inutile pour un carnet qui travaille en memoire. Sans elles, il compile en Rust pur et suffit a servir les requetes SPARQL des sections 4 a 9.

Deux differences structurelles avec le jumeau Python :

- **Pas d'introspection de version** : Rust ne dit pas a l'execution quelle version de crate tourne. Les versions affichees sont celles **epinglees** par les `:dep`, resolues par cargo ;
- **`Result` partout** : la ou rdflib leve une exception, la famille oxigraph retourne un `Result` — `NamedNode::new` peut echouer sur une IRI invalide, `Store::insert` sur une erreur de stockage, `Store::query` sur une syntaxe SPARQL invalide. Le carnet traite chaque cas explicitement.

Les `:dep` sont regroupes dans **une seule** cellule : chaque `:dep` declenche une compilation evcxr, les disperser ferait recompiler la pile a chaque cellule.

In [2]:
// Dependances pre-provisionnees (famille oxigraph) : chaque :dep declenche une
// compilation evcxr, les regrouper en tete evite les recompilations successives.
// oxigraph est charge SANS ses features par defaut : le backend RocksDB
// (oxrocksdb-sys, compile en C++) n'est pas necessaire pour un carnet en memoire.
:dep oxrdf = "0.3"
:dep oxttl = "0.2"
:dep oxrdfxml = "0.2"
:dep oxjsonld = "0.2"
:dep oxigraph = { version = "0.5", default-features = false }

#[allow(unused_imports)]
use oxrdf::{BlankNode, Graph, Literal, NamedNode, NamedOrBlankNode, Term, Triple};
#[allow(unused_imports)]
use oxttl::{NTriplesSerializer, TurtleSerializer};
#[allow(unused_imports)]
use oxrdfxml::RdfXmlSerializer;
#[allow(unused_imports)]
use oxjsonld::JsonLdSerializer;
#[allow(unused_imports)]
use oxigraph::model::{GraphName, Quad};
#[allow(unused_imports)]
use oxigraph::sparql::QueryResults;
#[allow(unused_imports)]
use oxigraph::store::Store;
#[allow(unused_imports)]
use std::collections::BTreeMap;

// La famille oxigraph ne fournit pas de constantes RDF/RDFS/XSD pre-declarees
// (contrairement a rdflib.namespace) : on fabrique les IRIs avec nn().
fn nn(s: &str) -> NamedNode { NamedNode::new(s).expect("IRI invalide") }
fn ex(l: &str) -> NamedNode { nn(&format!("http://example.org/{}", l)) }
fn rdf(l: &str) -> NamedNode { nn(&format!("http://www.w3.org/1999/02/22-rdf-syntax-ns#{}", l)) }
fn foaf(l: &str) -> NamedNode { nn(&format!("http://xmlns.com/foaf/0.1/{}", l)) }
fn xsd(l: &str) -> NamedNode { nn(&format!("http://www.w3.org/2001/XMLSchema#{}", l)) }
fn prov(l: &str) -> NamedNode { nn(&format!("http://www.w3.org/ns/prov#{}", l)) }
fn sdo(l: &str) -> NamedNode { nn(&format!("https://schema.org/{}", l)) }

// Litteraux types : rdflib deduit le type du type Python, oxrdf exige le XSD explicite.
fn txt(v: &str) -> Literal { Literal::new_simple_literal(v) }
fn dbl(v: f64) -> Literal { Literal::new_typed_literal(format!("{}", v), xsd("double")) }
fn int(v: i64) -> Literal { Literal::new_typed_literal(v.to_string(), xsd("integer")) }
fn date(v: &str) -> Literal { Literal::new_typed_literal(v, xsd("date")) }
fn gyear(v: &str) -> Literal { Literal::new_typed_literal(v, xsd("gYear")) }

// Un serialiseur par format : meme graphe, quatre syntaxes.
fn to_turtle(g: &Graph) -> String {
    let mut ser = TurtleSerializer::new()
        .with_prefix("ex", "http://example.org/").unwrap()
        .with_prefix("foaf", "http://xmlns.com/foaf/0.1/").unwrap()
        .with_prefix("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#").unwrap()
        .with_prefix("xsd", "http://www.w3.org/2001/XMLSchema#").unwrap()
        .for_writer(Vec::new());
    for t in g.iter() { ser.serialize_triple(t).unwrap(); }
    String::from_utf8(ser.finish().unwrap()).unwrap()
}
fn to_ntriples(g: &Graph) -> String {
    let mut ser = NTriplesSerializer::new().for_writer(Vec::new());
    for t in g.iter() { ser.serialize_triple(t).unwrap(); }
    String::from_utf8(ser.finish()).unwrap()
}
fn to_rdfxml(g: &Graph) -> String {
    let mut ser = RdfXmlSerializer::new()
        .with_prefix("ex", "http://example.org/").unwrap()
        .with_prefix("foaf", "http://xmlns.com/foaf/0.1/").unwrap()
        .with_prefix("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#").unwrap()
        .for_writer(Vec::new());
    for t in g.iter() { ser.serialize_triple(t).unwrap(); }
    String::from_utf8(ser.finish().unwrap()).unwrap()
}
fn to_jsonld(g: &Graph) -> String {
    let mut ser = JsonLdSerializer::new()
        .with_prefix("ex", "http://example.org/").unwrap()
        .with_prefix("foaf", "http://xmlns.com/foaf/0.1/").unwrap()
        .for_writer(Vec::new());
    for t in g.iter() {
        let t = t.into_owned();
        let q = Quad::new(t.subject, t.predicate, t.object, GraphName::DefaultGraph);
        ser.serialize_quad(&q).unwrap();
    }
    String::from_utf8(ser.finish().unwrap()).unwrap()
}

println!("Crates chargees : oxrdf 0.3, oxttl 0.2, oxrdfxml 0.2, oxjsonld 0.2, oxigraph 0.5 (memoire)");

Crates chargees : oxrdf 0.3, oxttl 0.2, oxrdfxml 0.2, oxjsonld 0.2, oxigraph 0.5 (memoire)


Verifions la pile chargee. Rust n'expose pas de numero de version a l'execution : les lignes ci-dessous rappellent les versions epinglees par les `:dep`, et l'etat **mesure** du support RDF-Star dans le modele (source `oxrdf-0.3.5/src/triple.rs`, lignes 170-178).

In [3]:
// Rust n'a pas d'introspection de version a l'execution : les versions ci-dessous
// sont celles EPINGLEES par les :dep de la cellule precedente, resolues par cargo.
println!("oxrdf    : 0.3 (0.3.5)");
println!("oxttl    : 0.2 (0.2.5)");
println!("oxrdfxml : 0.2 (0.2.5)");
println!("oxjsonld : 0.2 (0.2.7)");
println!("oxigraph : 0.5 (0.5.13, default-features = false : memoire seule)");
println!();

// Mesure de source (oxrdf-0.3.5/src/triple.rs:170-178) : le modele porte une
// quatrieme variante Term::Triple(Box<Triple>) pour les Quoted Triples RDF 1.2,
// mais elle n'est compilee QUE sous la feature cargo `rdf-12`, non activee ici.
println!("Support natif RDF-Star (Turtle-Star, QuotedTriple) : NON active.");
println!("Le modele oxrdf definit Term::Triple derriere la feature cargo `rdf-12` ;");
println!("ce carnet utilise la reification standard et les graphes nommes");
println!("comme alternatives fonctionnelles pour enseigner les memes concepts.");

oxrdf    : 0.3 (0.3.5)


oxttl    : 0.2 (0.2.5)


oxrdfxml : 0.2 (0.2.5)


oxjsonld : 0.2 (0.2.7)


oxigraph : 0.5 (0.5.13, default-features = false : memoire seule)


Support natif RDF-Star (Turtle-Star, QuotedTriple) : NON active.


Le modele oxrdf definit Term::Triple derriere la feature cargo `rdf-12` ;


ce carnet utilise la reification standard et les graphes nommes


comme alternatives fonctionnelles pour enseigner les memes concepts.


## 1. Le probleme : pourquoi annoter un triplet ?

Un triplet RDF — `sujet predicat objet` — est **atomique**. Il affirme un fait, sans dire *qui* l'affirme, *quand*, ni *avec quelle confiance*. Des que l'on veut representer :

- une **source** : « cette relation vient de LinkedIn »,
- une **confiance** : « 0,95, pas 1,0 »,
- une **validite** : « vrai de 2019 a 2021 »,
- une **croyance** : « Bob pense que… »,

il faut un moyen d'attacher des metadonnees **a un fait**, et non au sujet ou a l'objet du fait. Trois mecanismes existent, dans l'ordre historique :

| Mecanisme | Statut | Ce qu'il ajoute |
|---|---|---|
| **Reification** (`rdf:Statement`) | RDF 1.0, universel | 4 triplets par fait annote |
| **Graphes nommes** | RDF 1.1, universel | une IRI de contexte, annotable |
| **Quoted triples** (`<< s p o >>`) | RDF 1.2 | 0 triplet supplementaire, syntaxe neuve |

### 1.1 La reification classique : 4 triplets pour 1 assertion

RDF fournit depuis l'origine le vocabulaire `rdf:Statement` : un fait annote devient une **ressource** decrite par quatre triplets.

```turtle
# Le triplet original qu'on veut annoter :
#   ex:Bob foaf:knows ex:Carol .

# La reification : le fait devient un noeud
ex:statement1 a rdf:Statement ;
    rdf:subject   ex:Bob ;
    rdf:predicate foaf:knows ;
    rdf:object    ex:Carol .

# Les annotations se posent alors sur ce noeud
ex:statement1 ex:assertedBy ex:Alice ;
              ex:confidence 0.85 .
```

Sept triplets en tout : le fait original (1), sa decomposition (4), les annotations (2). Le cout est **structurel** : chaque annotation supplementaire s'ajoute a un noeud deja en place, mais la decomposition, elle, se repete pour chaque fait.

Voyons cela en pratique : la reification classique est la **seule** voie portable en RDF 1.0 et 1.1, donc celle que tout triple store comprend. C'est aussi celle que le jumeau Python utilise, rdflib 7.6 ne reconnaissant pas la syntaxe `<< … >>`.

In [4]:
// Le fait original
let mut g_reif = Graph::new();
g_reif.insert(&Triple::new(ex("Bob"), foaf("knows"), ex("Carol")));

// Reification classique : decomposer le triplet en quatre triplets rdf:Statement
let stmt = NamedOrBlankNode::from(ex("statement1"));
g_reif.insert(&Triple::new(stmt.clone(), rdf("type"), rdf("Statement")));
g_reif.insert(&Triple::new(stmt.clone(), rdf("subject"), ex("Bob")));
g_reif.insert(&Triple::new(stmt.clone(), rdf("predicate"), foaf("knows")));
g_reif.insert(&Triple::new(stmt.clone(), rdf("object"), ex("Carol")));

// Annotations : qui affirme ce fait, avec quel degre de confiance ?
g_reif.insert(&Triple::new(stmt.clone(), ex("assertedBy"), ex("Alice")));
g_reif.insert(&Triple::new(stmt.clone(), ex("confidence"), dbl(0.85)));

println!("Nombre total de triplets : {}", g_reif.len());
println!("  - Fait original : 1 triplet");
println!("  - Reification : 4 triplets");
println!("  - Annotations : 2 triplets");
println!("  - Total : 7 triplets pour annoter 1 fait");
println!();
println!("{}", to_turtle(&g_reif));

Nombre total de triplets : 7


  - Fait original : 1 triplet


  - Reification : 4 triplets


  - Annotations : 2 triplets


  - Total : 7 triplets pour annoter 1 fait


@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .


... (6 outputs left out)

	ex:confidence "0.85"^^xsd:double ;


	ex:assertedBy ex:Alice ;


	rdf:predicate foaf:knows ;


	rdf:object ex:Carol .


## Interpretation : cout de la reification

### Lecture des mesures

Le graphe compte **7 triplets** pour **1 fait annote** :

| Part | Triplets | Role |
|---|---|---|
| Fait original | 1 | l'assertion elle-meme, `ex:Bob foaf:knows ex:Carol` |
| Reification | 4 | `rdf:type`, `rdf:subject`, `rdf:predicate`, `rdf:object` |
| Annotations | 2 | `ex:assertedBy`, `ex:confidence` |

Deux consequences a garder en tete pour toute la suite :

- **Le cout est un multiple** : annoter *n* faits coute `n × (1 + 4 + k)` triplets, ou *k* est le nombre d'annotations. Un graphe de 10 000 relations annotees a 95 % de confiance pese au moins 60 000 triplets, dont 40 000 purement structurels ;
- **Le fait original reste asserté** : le triplet `ex:Bob foaf:knows ex:Carol` existe independamment de sa reification. On peut reifier un fait qu'on n'asserte pas — c'est ce que fait le fait 4 de la section 4.1, « affirme par Bob » mais jamais pose comme vrai.

Le Turtle produit par `to_turtle` montre la forme exacte : le noeud `ex:statement1` porte les cinq proprietes, et le fait original vit sa vie de son cote.

## 2. RDF 1.2 et les Quoted Triples

### Historique

Entre 2021 et 2023, le W3C a travaille sur **RDF-star** (renomme **RDF 1.2**), dont l'idee tient en une ligne : autoriser un triplet a occuper la place d'un sujet ou d'un objet.

### Syntaxe des Quoted Triples

```turtle
<< ex:Bob foaf:knows ex:Carol >> ex:assertedBy ex:Alice ;
                                 ex:confidence 0.85 .
```

Un seul triplet, aucune decomposition, le fait annote n'est pas forcement asserte. La syntaxe `<< … >>` s'appelle *quoted triple* ; la variante ou le triplet est aussi asserte s'ecrit `{| … |}` (annotation syntax).

### Etat cote Rust, mesure et non suppose

`oxrdf` 0.3.5 **connait** les quoted triples : la variante `Term::Triple(Box<Triple>)` existe dans l'enumeration `Term` (`oxrdf-0.3.5/src/triple.rs:170-178`). Mais elle est compilee **seulement** sous la feature cargo `rdf-12` :

```rust
#[cfg(feature = "rdf-12")]
Triple(Box<Triple>),
```

Feature desactivee, `Term` n'a que trois variantes (`NamedNode`, `BlankNode`, `Literal`) — c'est pour cela que les `match` de ce carnet compilent sans bras supplementaire.

**La chaine d'outils suit, en revanche — c'est un choix de carnet.** Mesure a la source (Cargo.toml des crates epinglees) : `oxttl` 0.2.5, `oxrdfxml` 0.2.5 et `oxjsonld` 0.2.7 declarent tous les trois `rdf-12 = ["oxrdf/rdf-12"]`. La pile Rust sait donc, feature activee, parser le Turtle 1.2 et ecrire des termes-triplets. Ce carnet ne l'active pas, pour deux raisons assumees :

- **coherence du modele** : activer `rdf-12` fait passer `Term` de trois a quatre variantes, donc tout `match` sur un `Term` du carnet change de forme ;
- **parite de contenu** : le jumeau Python ne peut pas suivre (rdflib 7.6 n'a ni `<< >>` ni `rdflib.term.Triple`) ; le carnet enseigne donc le chemin **portable**, celui qui traverse tous les formats et tous les consommateurs.

Un carnet dedie au RDF 1.2 natif en Rust a sa place — il traiterait `Term::Triple`, les annotations `{| … |}` et la serialisation Turtle 1.2 — mais ce n'est pas celui-ci.

### 2.1 Fonction utilitaire de reification

Repeter quatre `insert` a chaque fait est fastidieux et source d'erreurs : on ecrit la fonction une fois.

```rust
fn reify(
    g: &mut Graph,
    subject: &NamedOrBlankNode,
    predicate: &NamedNode,
    object: &Term,
    stmt: Option<NamedOrBlankNode>,
) -> NamedOrBlankNode
```

Trois choix de signature, propres a Rust :

- **`&mut Graph`** : Rust rend la mutation explicite. La ou `graph.add(...)` de rdflib modifie l'objet en place, ici le compilateur exige que l'appelant declare le graphe mutable, et interdit deux acces mutables simultanes ;
- **`&NamedOrBlankNode`, `&Term`** : le sujet et l'objet peuvent etre n'importe quel noeud RDF — IRI ou noeud blanc. `NamedOrBlankNode` est l'enumeration fermee a deux variantes qui rend ce contrat lisible ;
- **`Option<NamedOrBlankNode>`** : le Statement est un noeud blanc par defaut (`BlankNode::default()`), ou l'IRI fournie. `Option` remplace le `stmt_uri=None` de Python et rend l'absence explicite au type.

La fonction retourne le noeud Statement : l'appelant y attache ses annotations, ou le reifie a son tour pour une imbrication (section 3.2).

In [5]:
// Reifie un triplet : quatre triplets rdf:Statement, plus rien.
// En RDF-Star, cela serait simplement << subject predicate object >>.
// Le Statement est un BNode par defaut, ou l'IRI fournie en dernier argument.
fn reify(
    g: &mut Graph,
    subject: &NamedOrBlankNode,
    predicate: &NamedNode,
    object: &Term,
    stmt: Option<NamedOrBlankNode>,
) -> NamedOrBlankNode {
    let stmt = match stmt {
        Some(s) => s,
        None => NamedOrBlankNode::from(BlankNode::default()),
    };
    g.insert(&Triple::new(stmt.clone(), rdf("type"), rdf("Statement")));
    g.insert(&Triple::new(stmt.clone(), rdf("subject"), subject.clone()));
    g.insert(&Triple::new(stmt.clone(), rdf("predicate"), predicate.clone()));
    g.insert(&Triple::new(stmt.clone(), rdf("object"), object.clone()));
    stmt
}

// Demonstration : annoter "Bob connait Carol"
let mut g_demo = Graph::new();
g_demo.insert(&Triple::new(ex("Bob"), foaf("knows"), ex("Carol")));

let stmt = reify(&mut g_demo, &NamedOrBlankNode::from(ex("Bob")), &foaf("knows"), &Term::from(ex("Carol")), None);
g_demo.insert(&Triple::new(stmt.clone(), ex("assertedBy"), ex("Alice")));
g_demo.insert(&Triple::new(stmt.clone(), ex("confidence"), dbl(0.85)));
g_demo.insert(&Triple::new(stmt.clone(), ex("since"), date("2020-01-15")));

println!("Triplets charges : {}", g_demo.len());
println!("  - Fait original : 1");
println!("  - Reification (rdf:Statement) : 4");
println!("  - Annotations : 3");
println!("  - Total : {}", g_demo.len());
println!();
println!("=== Serialisation Turtle ===");
println!("{}", to_turtle(&g_demo));

Triplets charges : 8


  - Fait original : 1


  - Reification (rdf:Statement) : 4


  - Annotations : 3


  - Total : 8


=== Serialisation Turtle ===


... (8 outputs left out)

	ex:assertedBy ex:Alice ;


	ex:since "2020-01-15"^^xsd:date ;


	rdf:predicate foaf:knows ;


	rdf:object ex:Carol .


## Interpretation

### Lecture des mesures

Le graphe compte **8 triplets** : les 7 de la main sont la, plus `ex:since` (une annotation de plus). La forme est plus lisible qu'a la main parce que la fonction nomme l'intention — « reifier ce fait » — au lieu d'enumerer quatre triplets.

Deux points a noter :

- **Le Statement est un noeud blanc** : la fonction n'invente pas d'IRI. Un noeud blanc n'a pas d'identite globale ; deux noeuds blancs distincts restent distincts dans le graphe, mais leurs etiquettes (`_:N…`) sont regenerees a chaque serialisation ;
- **Le fait reste asserte** : la fonction ne pose que la decomposition. C'est l'appelant qui decide d'asserter le fait original, ou non. Cette separation est exactement ce que RDF 1.2 supprime — un quoted triple peut exister sans assertion.

## 3. Construction programmatique avec reification

Les sections precedentes montrent la mecanique sur un fait isole. En pratique, on construit des graphes : une boucle pose des faits, la fonction les reifie, les annotations s'ajoutent dans le meme mouvement.

### 3.1 Construction programmatique

Le meme graphe, construit en trois appels : assertion du fait, reification, annotations. En RDF 1.2, cela tiendrait en une ligne (`<< ex:Bob foaf:knows ex:Carol >> ex:assertedBy ex:Alice`), mais le resultat RDF 1.1 equivalent est bien celui-ci.

Observez ce que le type du Statement revele : un `NamedOrBlankNode` dont la variante est `BlankNode`. Le graphe ne connait pas de « huitieme noeud nomme » — il connait un noeud blanc qui porte quatre proprietes.

In [6]:
let mut g_prog = Graph::new();

// Asserter le fait original
g_prog.insert(&Triple::new(ex("Bob"), foaf("knows"), ex("Carol")));

// Reifier et annoter.
// En RDF-Star, cela serait : << ex:Bob foaf:knows ex:Carol >> ex:assertedBy ex:Alice .
let stmt = reify(&mut g_prog, &NamedOrBlankNode::from(ex("Bob")), &foaf("knows"), &Term::from(ex("Carol")), None);
g_prog.insert(&Triple::new(stmt.clone(), ex("assertedBy"), ex("Alice")));
g_prog.insert(&Triple::new(stmt.clone(), ex("confidence"), dbl(0.85)));
g_prog.insert(&Triple::new(stmt.clone(), ex("source"), txt("Interview 2024-03-15")));

let kind = match &stmt {
    NamedOrBlankNode::BlankNode(_) => "BNode",
    NamedOrBlankNode::NamedNode(_) => "NamedNode",
};

println!("Graphe construit : {} triplets", g_prog.len());
println!("  Type du Statement : {}", kind);
println!();
println!("=== Serialisation Turtle ===");
println!("{}", to_turtle(&g_prog));

Graphe construit : 8 triplets


  Type du Statement : BNode


=== Serialisation Turtle ===


@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .


@prefix foaf: <http://xmlns.com/foaf/0.1/> .


... (5 outputs left out)

	ex:source "Interview 2024-03-15" ;


	ex:confidence "0.85"^^xsd:double ;


	rdf:object ex:Carol ;


	rdf:subject ex:Bob .


## Interpretation

### Lecture des mesures

**8 triplets** : 1 fait + 4 structure + 3 annotations (`assertedBy`, `confidence`, `source`). Le graphe ne grandit pas autrement qu'a la main — la fonction n'ajoute rien, elle nomme seulement.

Ce que la serialisation N-Triples montrerait, et que le Turtle cache : l'etiquette du noeud blanc change a chaque execution, et sa forme depend de la bibliotheque. `oxrdf` ecrit l'empreinte en hexadecimal sans prefixe (`_:e82924c3…` dans la sortie de la section 3.1 de cette execution) ; rdflib prefixe la sienne d'un `N` (`_:Ne8e9…`). Deux consequences pratiques :

- **Ne jamais citer une etiquette de noeud blanc** dans une configuration, un test ou une requete : elle n'est stable ni entre deux executions, ni entre deux bibliotheques ;
- **Pour une identite stable**, passer un nom : `reify(..., Some(NamedOrBlankNode::from(ex("statement1"))))`. C'est ce que fait la section 1, ou l'IRI `ex:statement1` est explicite.

### 3.2 Annotations imbriquees (multi-niveaux)

Un fait peut lui-meme etre l'objet d'une croyance, qu'une troisieme personne rapporte :

```text
Niveau 1 : Carol connait Dave                       (1 triplet, asserte)
Niveau 2 : Bob croit ce fait                        (reification + 1 annotation)
Niveau 3 : Alice rapporte la croyance de Bob        (reification du Statement de niveau 2)
```

En RDF 1.2, l'imbrication est directe : `<< << ex:Carol foaf:knows ex:Dave >> ex:believedBy ex:Bob >> ex:reportedBy ex:Alice`. En RDF 1.1, on reifie le Statement de niveau 2, qui est un noeud comme un autre : la fonction `reify` accepte donc `&stmt_level1` en position de sujet.

In [7]:
let mut g_nested = Graph::new();

// Niveau 1 : Carol connait Dave (fait de base)
g_nested.insert(&Triple::new(ex("Carol"), foaf("knows"), ex("Dave")));

// Niveau 2 : Bob croit ce fait
// En RDF-Star : << ex:Carol foaf:knows ex:Dave >> ex:believedBy ex:Bob .
let stmt_level1 = reify(&mut g_nested, &NamedOrBlankNode::from(ex("Carol")), &foaf("knows"), &Term::from(ex("Dave")), None);
g_nested.insert(&Triple::new(stmt_level1.clone(), ex("believedBy"), ex("Bob")));

// Niveau 3 : Alice rapporte la croyance de Bob
// En RDF-Star : << << ... >> ex:believedBy ex:Bob >> ex:reportedBy ex:Alice .
// Avec la reification, on reifie le Statement de niveau 2.
let stmt_level2 = reify(&mut g_nested, &stmt_level1, &ex("believedBy"), &Term::from(ex("Bob")), None);
g_nested.insert(&Triple::new(stmt_level2.clone(), ex("reportedBy"), ex("Alice")));
g_nested.insert(&Triple::new(stmt_level2.clone(), ex("reportDate"), date("2024-06-01")));

println!("Triplets charges : {}", g_nested.len());
println!("  - Fait de base : 1");
println!("  - Reification niveau 1 : 4 + 1 annotation");
println!("  - Reification niveau 2 : 4 + 2 annotations");
println!("  - Total : {}", g_nested.len());
println!();
println!("=== Serialisation Turtle ===");
println!("{}", to_turtle(&g_nested));

Triplets charges : 12


  - Fait de base : 1


  - Reification niveau 1 : 4 + 1 annotation


  - Reification niveau 2 : 4 + 2 annotations


  - Total : 12


=== Serialisation Turtle ===


... (12 outputs left out)

	a rdf:Statement ;


	ex:believedBy ex:Bob ;


	rdf:predicate foaf:knows ;


	rdf:object ex:Dave .


## Interpretation : imbrication

### Lecture des mesures

**12 triplets** pour un fait de base, decomposition exacte du budget :

| Niveau | Triplets | Detail |
|---|---|---|
| Fait de base | 1 | `ex:Carol foaf:knows ex:Dave` |
| Reification 1 | 4 + 1 | 4 de structure, 1 annotation (`believedBy`) |
| Reification 2 | 4 + 2 | 4 de structure, 2 annotations (`reportedBy`, `reportDate`) |
| **Total** | **12** | |

La croissance est **lineaire en profondeur, structurelle en largeur** : chaque niveau coute 4 triplets fixes plus ses annotations. Une chaine de croyances sur 5 niveaux coute deja 21 triplets pour un seul fait de base (1 fait + 5 × 4 triplets de structure, sans aucune annotation) — c'est le principal argument en faveur des quoted triples, qui rameneraient ce graphe a 3 triplets : le fait de base et les deux annotations qui citent un triplet au lieu de le decomposer.

Le Turtle produit est instructif : le noeud de niveau 2 apparait **imbrique** (`rdf:subject [ a rdf:Statement ; … ]`), parce que le serialiseur Turtle regroupe les noeuds blancs sans reference externe. La structure est identique, seule l'ecriture differe.

## 4. Interroger les annotations avec SPARQL

Annoter ne sert a rien si l'on ne peut pas interroger. La reification se prete a une jointure fixe : un `rdf:Statement` est lie a son fait par `rdf:subject`, `rdf:predicate`, `rdf:object`, et a ses annotations par n'importe quel predicat.

### Comparaison des syntaxes

| Besoin | SPARQL 1.1 (reification) | SPARQL-Star (RDF 1.2) |
|---|---|---|
| Le fait annote | 4 triplets de jointure | un *triple pattern* imbrique |
| Filtrer sur la confiance | `?stmt ex:confidence ?c . FILTER(?c >= 0.6)` | identique |
| Le fait **non asserte** | visible (il n'est pas dans le graphe) | visible aussi |
| Joindre sujet et objet | `rdf:subject` / `rdf:object` explicites | `<< ?s ?p ?o >>` |

Cote Rust, `oxrdf::Graph` ne sait pas interroger : il ne fait que contenir. Le moteur SPARQL est `oxigraph::Store`, une base en memoire. Les deux crates partagent le **meme** modele de donnees — `oxigraph` 0.5.13 epingle `oxrdf = "=0.3.5"` — donc le passage d'un triplet de l'un a l'autre se fait sans conversion de type, par une simple construction de `Quad`.

### 4.1 Construire un graphe de connaissances annote

Quatre personnes, quatre relations, quatre annotations differentes :

| Fait | Confiance | Source | Depuis | Asserte ? |
|---|---|---|---|---|
| Alice connait Bob | 0,95 | LinkedIn | 2019-03-01 | oui |
| Bob connait Carol | 0,60 | Twitter | 2021-07-15 | oui |
| Carol connait Dave | 0,30 | Rumeur | — | oui |
| Alice connait Dave | 0,50 | (affirme par Bob) | — | **non** |

La quatrieme ligne est le cas interessant : le fait `ex:Alice foaf:knows ex:Dave` **n'est pas** dans le graphe. Seule sa reification existe, avec `ex:assertedBy ex:Bob`. C'est la difference entre « le monde est ainsi » et « Bob pense que le monde est ainsi » — et c'est exactement ce qu'une reification permet de representer en RDF 1.1.

In [8]:
let mut g_kg = Graph::new();

// === Personnes ===
for (name, uri) in [
    ("Alice Dupont", "Alice"),
    ("Bob Martin", "Bob"),
    ("Carol Laurent", "Carol"),
    ("Dave Petit", "Dave"),
] {
    g_kg.insert(&Triple::new(ex(uri), rdf("type"), foaf("Person")));
    g_kg.insert(&Triple::new(ex(uri), foaf("name"), txt(name)));
}

// === Fait 1 : Alice connait Bob (haute confiance, source LinkedIn) ===
// RDF-Star : << ex:Alice foaf:knows ex:Bob >> ex:confidence "0.95"^^xsd:double .
g_kg.insert(&Triple::new(ex("Alice"), foaf("knows"), ex("Bob")));
let stmt1 = reify(&mut g_kg, &NamedOrBlankNode::from(ex("Alice")), &foaf("knows"), &Term::from(ex("Bob")), None);
g_kg.insert(&Triple::new(stmt1.clone(), ex("confidence"), dbl(0.95)));
g_kg.insert(&Triple::new(stmt1.clone(), ex("source"), txt("LinkedIn")));
g_kg.insert(&Triple::new(stmt1.clone(), ex("since"), date("2019-03-01")));

// === Fait 2 : Bob connait Carol (confiance moyenne, source Twitter) ===
g_kg.insert(&Triple::new(ex("Bob"), foaf("knows"), ex("Carol")));
let stmt2 = reify(&mut g_kg, &NamedOrBlankNode::from(ex("Bob")), &foaf("knows"), &Term::from(ex("Carol")), None);
g_kg.insert(&Triple::new(stmt2.clone(), ex("confidence"), dbl(0.60)));
g_kg.insert(&Triple::new(stmt2.clone(), ex("source"), txt("Twitter")));
g_kg.insert(&Triple::new(stmt2.clone(), ex("since"), date("2021-07-15")));

// === Fait 3 : Carol connait Dave (basse confiance, source rumeur) ===
g_kg.insert(&Triple::new(ex("Carol"), foaf("knows"), ex("Dave")));
let stmt3 = reify(&mut g_kg, &NamedOrBlankNode::from(ex("Carol")), &foaf("knows"), &Term::from(ex("Dave")), None);
g_kg.insert(&Triple::new(stmt3.clone(), ex("confidence"), dbl(0.30)));
g_kg.insert(&Triple::new(stmt3.clone(), ex("source"), txt("Rumeur")));

// === Fait 4 : Alice connait Dave (NON asserte, seulement affirme par Bob) ===
// On n'insere PAS (Alice, foaf:knows, Dave) : seule la reification existe.
let stmt4 = reify(&mut g_kg, &NamedOrBlankNode::from(ex("Alice")), &foaf("knows"), &Term::from(ex("Dave")), None);
g_kg.insert(&Triple::new(stmt4.clone(), ex("assertedBy"), ex("Bob")));
g_kg.insert(&Triple::new(stmt4.clone(), ex("confidence"), dbl(0.50)));

println!("Graphe de connaissances : {} triplets", g_kg.len());
println!();
println!("{}", to_turtle(&g_kg));

Graphe de connaissances : 37 triplets


@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .


@prefix foaf: <http://xmlns.com/foaf/0.1/> .


@prefix ex: <http://example.org/> .


ex:Carol foaf:knows ex:Dave ;


... (32 outputs left out)

	a rdf:Statement ;


	ex:confidence "0.3"^^xsd:double ;


	ex:source "Rumeur" ;


	rdf:predicate foaf:knows .


## Interpretation : graphe de connaissances annote

### Lecture des mesures — decomposition des 37 triplets

| Part | Triplets | Calcul |
|---|---|---|
| Personnes | 8 | 4 personnes × (`rdf:type` + `foaf:name`) |
| Fait 1 (Alice→Bob, 0,95) | 8 | 1 assertion + 4 structure + 3 annotations |
| Fait 2 (Bob→Carol, 0,60) | 8 | 1 + 4 + 3 |
| Fait 3 (Carol→Dave, 0,30) | 7 | 1 + 4 + 2 (pas de `since`) |
| Fait 4 (Alice→Dave, **non asserte**) | 6 | 4 structure + 2 annotations, **sans** assertion |
| **Total** | **37** | |

Deux enseignements :

- **26 des 37 triplets** (70 %) sont de la structure ou de l'annotation — 16 de structure (4 par fait reifie) plus 10 d'annotations : seule une poignee porte des faits metier. Un graphe annote est majoritairement descriptif ;
- **Le fait 4 coute 6 triplets et n'affirme rien.** C'est le prix de l'expressivite en RDF 1.1 : dire « Bob affirme que Alice connait Dave » sans dire « Alice connait Dave ». Les quoted triples rameneraient ces 6 triplets a 1.

En Rust, la construction est identique a la main `reify(&mut g_kg, …)` repetee quatre fois ; ce qui change par rapport au jumeau Python, c'est que le compilateur refuse tout oubli de `&mut` — un graphe ne se modifie pas par surprise.

### 4.2 Requetes SPARQL sur les reifications

La requete joint le Statement aux trois composantes du fait annote, puis recupere la confiance obligatoire et les deux annotations optionnelles (`OPTIONAL`, car tous les faits n'ont pas de `source` ni de `since`).

Point d'attention Rust : le `Store` se construit a partir du `Graph` par la fonction `store_of`, qui recopie chaque triplet en `Quad` dans le graphe par defaut. C'est la seule copie du carnet — `oxigraph` ne peut pas emprunter un `Graph` deja construit.

In [9]:
// Pont Graph -> Store : oxigraph 0.5.13 epingle oxrdf =0.3.5, donc les deux
// crates partagent le meme modele de donnees et un Triple se convertit sans copie.
fn store_of(g: &Graph) -> Store {
    let store = Store::new().expect("store memoire");
    for t in g.iter() {
        let t = t.into_owned();
        let q = Quad::new(t.subject, t.predicate, t.object, GraphName::DefaultGraph);
        store.insert(&q).expect("insertion");
    }
    store
}

// Lectures de Term : nom local d'une IRI, valeur d'un litteral, valeur numerique.
fn local(t: &Term) -> String {
    let s = t.to_string();
    let s = s.trim_start_matches('<').trim_end_matches('>');
    s.rsplit('/').next().unwrap_or(s).to_string()
}
fn value(t: &Term) -> String {
    match t {
        Term::Literal(l) => l.value().to_string(),
        other => local(other),
    }
}
fn num(t: &Term) -> f64 {
    match t {
        Term::Literal(l) => l.value().parse().unwrap_or(f64::NAN),
        _ => f64::NAN,
    }
}

// Requete 1 : toutes les relations annotees avec confiance.
// En SPARQL-Star, on ecrirait : << ?person1 foaf:knows ?person2 >> ex:confidence ?confidence .
// Avec la reification, on joint sur rdf:subject, rdf:predicate, rdf:object.
let store_kg = store_of(&g_kg);

let query_all = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?person1 ?person2 ?confidence ?source ?since
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?person1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?person2 ;
          ex:confidence ?confidence .
    OPTIONAL { ?stmt ex:source ?source . }
    OPTIONAL { ?stmt ex:since ?since . }
}
ORDER BY DESC(?confidence)
"#;

println!("=== Relations annotees (triees par confiance) ===");
println!("{:<15} {:<15} {:>10} {:<12} {}", "Personne 1", "Personne 2", "Confiance", "Source", "Depuis");
println!("{}", "-".repeat(70));
if let QueryResults::Solutions(solutions) = store_kg.query(query_all).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let p1 = local(sol.get("person1").expect("p1"));
        let p2 = local(sol.get("person2").expect("p2"));
        let conf = num(sol.get("confidence").expect("confiance"));
        let src = sol.get("source").map(value).unwrap_or_else(|| "-".to_string());
        let since = sol.get("since").map(value).unwrap_or_else(|| "-".to_string());
        println!("{:<15} {:<15} {:>10.2} {:<12} {}", p1, p2, conf, src, since);
    }
};

=== Relations annotees (triees par confiance) ===


Personne 1      Personne 2       Confiance Source       Depuis


----------------------------------------------------------------------


Alice           Bob                   0.95 LinkedIn     2019-03-01


Bob             Carol                 0.60 Twitter      2021-07-15


Alice           Dave                  0.50 -            -


Carol           Dave                  0.30 Rumeur       -


### 4.3 Filtrer par score de confiance

Le seuil `0.60` s'ecrit **type** dans la requete : `"0.60"^^xsd:double`. Sans le type, la comparaison porterait sur des chaines, et `"0.6"` serait superieur a `"0.60"` en ordre lexicographique. La reification ne rend pas ce piege plus frequent, mais elle le rend visible : c'est le graphe qui porte le type, pas la variable.

In [10]:
// Requete : relations a haute confiance (>= 0.60)
// En SPARQL-Star : << ?p1 foaf:knows ?p2 >> ex:confidence ?c . FILTER(?c >= 0.60)
let query_high = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?person1 ?person2 ?confidence ?source
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?person1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?person2 ;
          ex:confidence ?confidence .
    FILTER (?confidence >= "0.60"^^xsd:double)
    OPTIONAL { ?stmt ex:source ?source . }
}
ORDER BY DESC(?confidence)
"#;

// Requete : relations a basse confiance (< 0.60)
let query_low = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?person1 ?person2 ?confidence ?source
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?person1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?person2 ;
          ex:confidence ?confidence .
    FILTER (?confidence < "0.60"^^xsd:double)
    OPTIONAL { ?stmt ex:source ?source . }
}
ORDER BY ?confidence
"#;

let mut fiables = 0;
let mut incertaines = 0;

println!("=== Relations FIABLES (confiance >= 0.60) ===");
if let QueryResults::Solutions(solutions) = store_kg.query(query_high).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let p1 = local(sol.get("person1").expect("p1"));
        let p2 = local(sol.get("person2").expect("p2"));
        let conf = num(sol.get("confidence").expect("confiance"));
        let src = sol.get("source").map(value).unwrap_or_else(|| "-".to_string());
        println!("  {} connait {} (confiance={:.2}, source={})", p1, p2, conf, src);
        fiables += 1;
    }
}

println!();
println!("=== Relations INCERTAINES (confiance < 0.60) ===");
if let QueryResults::Solutions(solutions) = store_kg.query(query_low).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let p1 = local(sol.get("person1").expect("p1"));
        let p2 = local(sol.get("person2").expect("p2"));
        let conf = num(sol.get("confidence").expect("confiance"));
        let src = sol.get("source").map(value).unwrap_or_else(|| "-".to_string());
        println!("  {} connait {} (confiance={:.2}, source={})", p1, p2, conf, src);
        incertaines += 1;
    }
}

println!();
println!("Bilan du filtrage : {} fiables, {} incertaines", fiables, incertaines);

=== Relations FIABLES (confiance >= 0.60) ===


  Alice connait Bob (confiance=0.95, source=LinkedIn)


  Bob connait Carol (confiance=0.60, source=Twitter)


=== Relations INCERTAINES (confiance < 0.60) ===


  Carol connait Dave (confiance=0.30, source=Rumeur)


  Alice connait Dave (confiance=0.50, source=-)


Bilan du filtrage : 2 fiables, 2 incertaines


## Interpretation : filtrage par confiance

### Lecture des mesures

Le seuil coupe le graphe en deux, exactement :

| Ensemble | Relations | Confiances |
|---|---|---|
| Fiables (≥ 0,60) | 2 | Alice→Bob 0,95 ; Bob→Carol 0,60 |
| Incertaines (< 0,60) | 2 | Alice→Dave 0,50 ; Carol→Dave 0,30 |

Le cas limite est instructif : **Bob→Carol a exactement 0,60**, et tombe du cote fiable parce que le seuil est `>=`. La meme relation a `0.599999` serait incertaine — d'ou l'interet de typer les litteraux et de tester un seuil avec une valeur qui le touche.

Les deux requetes different par leur tri : `ORDER BY DESC(?confidence)` pour les fiables, `ORDER BY ?confidence` pour les incertaines. En SPARQL, un tri sur un litteral type `xsd:double` compare des nombres ; un tri sur un litteral simple comparerait des chaines.

## 5. Cas d'usage concrets

Trois usages reviennent dans tous les graphes de production :

1. **Provenance** — d'ou vient ce fait ? (section 5.1)
2. **Temporalite** — quand ce fait est-il vrai ? (section 5.2)
3. **Fusion multi-sources** — deux sources se contredisent, laquelle croire ? (section 5.3)

Chacun s'ecrit avec la meme mecanique ; ce qui change est le vocabulaire d'annotation.

### 5.1 Provenance et attribution

Le vocabulaire **PROV-O** (`http://www.w3.org/ns/prov#`) normalise l'attribution : `prov:wasDerivedFrom` designe la source, `prov:generatedAtTime` l'instant de production. Il se combine avec des predicats metier (`ex:method`, `ex:accuracy`) sans conflit.

```turtle
ex:Paris schema:population 2161000 .
[] a rdf:Statement ;
   rdf:subject ex:Paris ; rdf:predicate schema:population ; rdf:object 2161000 ;
   prov:wasDerivedFrom ex:INSEE_2023 ;
   prov:generatedAtTime "2023-12-01"^^xsd:date ;
   ex:method "Recensement" .
```

C'est le cas d'usage qui a motive RDF-star : dans un graphe de donnees ouvertes, **un fait sans source n'est pas un fait**, c'est une rumeur.

In [11]:
let mut g_prov = Graph::new();

// Fait 1 : Paris a une population de 2,1 millions
let pop_value = int(2161000);
g_prov.insert(&Triple::new(ex("Paris"), sdo("population"), pop_value.clone()));

// Provenance du fait (reification)
let stmt_pop = reify(&mut g_prov, &NamedOrBlankNode::from(ex("Paris")), &sdo("population"), &Term::from(pop_value), None);
g_prov.insert(&Triple::new(stmt_pop.clone(), prov("wasDerivedFrom"), ex("INSEE_2023")));
g_prov.insert(&Triple::new(stmt_pop.clone(), prov("generatedAtTime"), date("2023-12-01")));
g_prov.insert(&Triple::new(stmt_pop.clone(), ex("method"), txt("Recensement")));
g_prov.insert(&Triple::new(stmt_pop.clone(), ex("accuracy"), txt("haute")));

// Fait 2 : Paris a une superficie de 105.4 km2
let area_value = dbl(105.4);
g_prov.insert(&Triple::new(ex("Paris"), sdo("area"), area_value.clone()));

let stmt_area = reify(&mut g_prov, &NamedOrBlankNode::from(ex("Paris")), &sdo("area"), &Term::from(area_value), None);
g_prov.insert(&Triple::new(stmt_area.clone(), prov("wasDerivedFrom"), ex("IGN_2023")));
g_prov.insert(&Triple::new(stmt_area.clone(), prov("generatedAtTime"), date("2023-06-15")));
g_prov.insert(&Triple::new(stmt_area.clone(), ex("method"), txt("Mesure cadastrale")));

println!("Graphe de provenance : {} triplets", g_prov.len());
println!();

// Requete : quels faits, avec quelle source ?
let query_prov = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX schema: <https://schema.org/>

SELECT ?subject ?property ?value ?source ?date ?method
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?subject ;
          rdf:predicate ?property ;
          rdf:object ?value ;
          prov:wasDerivedFrom ?source .
    OPTIONAL { ?stmt prov:generatedAtTime ?date . }
    OPTIONAL { ?stmt ex:method ?method . }
}
"#;

println!("=== Faits avec provenance ===");
println!("{:<10} {:<20} {:<12} {:<15} {:<12} {}", "Sujet", "Propriete", "Valeur", "Source", "Date", "Methode");
println!("{}", "-".repeat(85));
let store_prov = store_of(&g_prov);
if let QueryResults::Solutions(solutions) = store_prov.query(query_prov).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let subj = local(sol.get("subject").expect("sujet"));
        let prop = local(sol.get("property").expect("propriete"));
        let val = value(sol.get("value").expect("valeur"));
        let src = local(sol.get("source").expect("source"));
        let d = sol.get("date").map(value).unwrap_or_else(|| "-".to_string());
        let method = sol.get("method").map(value).unwrap_or_else(|| "-".to_string());
        println!("{:<10} {:<20} {:<12} {:<15} {:<12} {}", subj, prop, val, src, d, method);
    }
};

Graphe de provenance : 17 triplets


=== Faits avec provenance ===


Sujet      Propriete            Valeur       Source          Date         Methode


-------------------------------------------------------------------------------------


Paris      area                 105.4        IGN_2023        2023-06-15   Mesure cadastrale


Paris      population           2161000      INSEE_2023      2023-12-01   Recensement


## Interpretation : provenance

### Lecture des mesures

**17 triplets** pour deux faits de Paris :

| Part | Triplets | Calcul |
|---|---|---|
| Faits assertes | 2 | `schema:population`, `schema:area` |
| Structure de reification | 8 | 2 × 4 |
| Annotations | 7 | population : 4 (`wasDerivedFrom`, `generatedAtTime`, `method`, `accuracy`) ; area : 3 |
| **Total** | **17** | |

La requete rend les deux faits avec leur source et leur methode. Deux details de typage :

- **`2161000` est un `xsd:integer`**, `105.4` un `xsd:double` : `Literal::new_typed_literal` exige le type, la ou Python le deduisait du type de l'objet ;
- **La colonne `Valeur`** affiche la valeur du litteral (`value()`), pas sa forme serialisee : c'est la difference entre lire un nombre et lire `"2161000"^^xsd:integer`.

### 5.2 Annotations temporelles

Un fait peut etre vrai sur un **intervalle**. Pour un parcours professionnel, chaque emploi porte `ex:startDate`, `ex:endDate`, `ex:role`, `ex:status` — et **seul l'emploi courant est asserte** :

```text
Marie schema:worksFor ex:Sorbonne .        # asserte : elle travaille la
[] a rdf:Statement ;                        # CNRS : termine, non asserte
   rdf:subject ex:Marie ; rdf:predicate schema:worksFor ; rdf:object ex:CNRS ;
   ex:endDate "2020-08-31"^^xsd:date .
```

Sans reification, on ne pourrait pas representer un emploi passe sans affirmer qu'il est encore vrai. C'est l'argument le plus convaincant en faveur des annotations de triplets dans les referentiels historiques.

In [12]:
let mut g_temporal = Graph::new();

// Personnes et organisations
for (name, uri, rdf_type) in [
    ("Marie", "Marie", "Person"),
    ("Sorbonne", "Sorbonne", "Organization"),
    ("CNRS", "CNRS", "Organization"),
    ("MIT", "MIT", "Organization"),
] {
    g_temporal.insert(&Triple::new(ex(uri), rdf("type"), sdo(rdf_type)));
    g_temporal.insert(&Triple::new(ex(uri), sdo("name"), txt(name)));
}

// Emploi actuel : Marie -> Sorbonne (asserte)
g_temporal.insert(&Triple::new(ex("Marie"), sdo("worksFor"), ex("Sorbonne")));
let stmt_sorb = reify(&mut g_temporal, &NamedOrBlankNode::from(ex("Marie")), &sdo("worksFor"), &Term::from(ex("Sorbonne")), None);
g_temporal.insert(&Triple::new(stmt_sorb.clone(), ex("startDate"), date("2020-09-01")));
g_temporal.insert(&Triple::new(stmt_sorb.clone(), ex("role"), txt("Professeure")));
g_temporal.insert(&Triple::new(stmt_sorb.clone(), ex("status"), txt("actif")));

// Emploi precedent 1 : Marie -> CNRS (non asserte, termine)
let stmt_cnrs = reify(&mut g_temporal, &NamedOrBlankNode::from(ex("Marie")), &sdo("worksFor"), &Term::from(ex("CNRS")), None);
g_temporal.insert(&Triple::new(stmt_cnrs.clone(), ex("startDate"), date("2015-01-15")));
g_temporal.insert(&Triple::new(stmt_cnrs.clone(), ex("endDate"), date("2020-08-31")));
g_temporal.insert(&Triple::new(stmt_cnrs.clone(), ex("role"), txt("Chercheuse")));
g_temporal.insert(&Triple::new(stmt_cnrs.clone(), ex("status"), txt("termine")));

// Emploi precedent 2 : Marie -> MIT (non asserte, termine)
let stmt_mit = reify(&mut g_temporal, &NamedOrBlankNode::from(ex("Marie")), &sdo("worksFor"), &Term::from(ex("MIT")), None);
g_temporal.insert(&Triple::new(stmt_mit.clone(), ex("startDate"), date("2012-09-01")));
g_temporal.insert(&Triple::new(stmt_mit.clone(), ex("endDate"), date("2014-12-31")));
g_temporal.insert(&Triple::new(stmt_mit.clone(), ex("role"), txt("Post-doc")));
g_temporal.insert(&Triple::new(stmt_mit.clone(), ex("status"), txt("termine")));

println!("Graphe temporel : {} triplets", g_temporal.len());
println!();

// Requete : parcours professionnel chronologique
let query_career = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX schema: <https://schema.org/>

SELECT ?orgName ?role ?startDate ?endDate ?status
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ex:Marie ;
          rdf:predicate schema:worksFor ;
          rdf:object ?org ;
          ex:role ?role ;
          ex:startDate ?startDate ;
          ex:status ?status .
    OPTIONAL { ?stmt ex:endDate ?endDate . }
    ?org schema:name ?orgName .
}
ORDER BY ?startDate
"#;

println!("=== Parcours professionnel de Marie ===");
println!("{:<15} {:<15} {:<12} {:<12} {}", "Organisation", "Role", "Debut", "Fin", "Statut");
println!("{}", "-".repeat(65));
let store_temporal = store_of(&g_temporal);
if let QueryResults::Solutions(solutions) = store_temporal.query(query_career).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let org = value(sol.get("orgName").expect("organisation"));
        let role = value(sol.get("role").expect("role"));
        let start = value(sol.get("startDate").expect("debut"));
        let end = sol.get("endDate").map(value).unwrap_or_else(|| "en cours".to_string());
        let status = value(sol.get("status").expect("statut"));
        println!("{:<15} {:<15} {:<12} {:<12} {}", org, role, start, end, status);
    }
};

Graphe temporel : 32 triplets


=== Parcours professionnel de Marie ===


Organisation    Role            Debut        Fin          Statut


-----------------------------------------------------------------


MIT             Post-doc        2012-09-01   2014-12-31   termine


CNRS            Chercheuse      2015-01-15   2020-08-31   termine


Sorbonne        Professeure     2020-09-01   en cours     actif


## Interpretation : annotations temporelles

### Lecture des mesures

**32 triplets** pour un parcours de trois emplois :

| Part | Triplets | Calcul |
|---|---|---|
| Entites (Marie, Sorbonne, CNRS, MIT) | 8 | 4 × (`rdf:type` + `schema:name`) |
| Emploi courant asserte | 1 | `Marie schema:worksFor Sorbonne` |
| Sorbonne (reification) | 7 | 4 structure + 3 annotations (`startDate`, `role`, `status`) |
| CNRS (non asserte) | 8 | 4 structure + 4 annotations (dont `endDate`) |
| MIT (non asserte) | 8 | 4 structure + 4 annotations |
| **Total** | **32** | |

La requete rend les trois emplois dans l'ordre chronologique, avec `en cours` pour l'emploi dont `ex:endDate` est absent (`OPTIONAL` + valeur par defaut). Deux emplois sur trois ne sont **pas** assertes dans le graphe : ils n'existent que comme reifications. Un consommateur qui interroge `?p schema:worksFor ?o` sans jointure sur `rdf:Statement` ne voit donc que Sorbonne — c'est voulu, et c'est ce qui distingue « travaille actuellement pour » de « a travaille pour ».

### 5.3 Fusion multi-sources avec scores de confiance

Trois sources donnent trois populations differentes pour Lyon. Le graphe les porte **toutes** — aucune n'est ecrasee — chacune avec sa confiance et son annee. La decision (« quelle valeur retenir ») est une **requete**, pas une ecriture.

C'est l'inverse d'une base de donnees classique, ou la derniere ecriture gagne : ici, le graphe conserve le desaccord, et l'application choisit sa politique (`ORDER BY DESC(?confidence) LIMIT 1`, ou une moyenne ponderee, ou une revue humaine).

In [13]:
// Formatage a la francaise d'un entier : separateur de milliers par virgule.
fn thousands(v: i64) -> String {
    let s = v.abs().to_string();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (s.len() - i) % 3 == 0 { out.push(','); }
        out.push(ch);
    }
    if v < 0 { format!("-{}", out) } else { out }
}

let mut g_multi = Graph::new();
g_multi.insert(&Triple::new(ex("Lyon"), rdf("type"), sdo("City")));
g_multi.insert(&Triple::new(ex("Lyon"), sdo("name"), txt("Lyon")));

// Source 1 : INSEE (haute confiance)
// RDF-Star : << ex:Lyon schema:population "522250"^^xsd:integer >> ex:source "INSEE" .
let pop_insee = int(522250);
let stmt_insee = reify(&mut g_multi, &NamedOrBlankNode::from(ex("Lyon")), &sdo("population"), &Term::from(pop_insee), None);
g_multi.insert(&Triple::new(stmt_insee.clone(), ex("source"), txt("INSEE")));
g_multi.insert(&Triple::new(stmt_insee.clone(), ex("confidence"), dbl(0.95)));
g_multi.insert(&Triple::new(stmt_insee.clone(), ex("year"), gyear("2023")));

// Source 2 : Wikipedia (confiance moyenne)
let pop_wiki = int(516092);
let stmt_wiki = reify(&mut g_multi, &NamedOrBlankNode::from(ex("Lyon")), &sdo("population"), &Term::from(pop_wiki), None);
g_multi.insert(&Triple::new(stmt_wiki.clone(), ex("source"), txt("Wikipedia")));
g_multi.insert(&Triple::new(stmt_wiki.clone(), ex("confidence"), dbl(0.70)));
g_multi.insert(&Triple::new(stmt_wiki.clone(), ex("year"), gyear("2021")));

// Source 3 : estimation locale (basse confiance)
let pop_mairie = int(530000);
let stmt_mairie = reify(&mut g_multi, &NamedOrBlankNode::from(ex("Lyon")), &sdo("population"), &Term::from(pop_mairie), None);
g_multi.insert(&Triple::new(stmt_mairie.clone(), ex("source"), txt("Estimation mairie")));
g_multi.insert(&Triple::new(stmt_mairie.clone(), ex("confidence"), dbl(0.40)));
g_multi.insert(&Triple::new(stmt_mairie.clone(), ex("year"), gyear("2024")));

println!("Graphe multi-sources : {} triplets", g_multi.len());
println!();

// Requete : toutes les valeurs avec leurs sources
let query_sources = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX schema: <https://schema.org/>

SELECT ?pop ?source ?confidence ?year
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ex:Lyon ;
          rdf:predicate schema:population ;
          rdf:object ?pop ;
          ex:source ?source ;
          ex:confidence ?confidence ;
          ex:year ?year .
}
ORDER BY DESC(?confidence)
"#;

println!("=== Population de Lyon selon differentes sources ===");
println!("{:<20} {:>12} {:>10} {:>6}", "Source", "Population", "Confiance", "Annee");
println!("{}", "-".repeat(52));
let store_multi = store_of(&g_multi);
if let QueryResults::Solutions(solutions) = store_multi.query(query_sources).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let pop = num(sol.get("pop").expect("population")) as i64;
        let src = value(sol.get("source").expect("source"));
        let conf = num(sol.get("confidence").expect("confiance"));
        let year = value(sol.get("year").expect("annee"));
        println!("{:<20} {:>12} {:>10.2} {:>6}", src, thousands(pop), conf, year);
    }
}

// Valeur la plus fiable
let query_best = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX schema: <https://schema.org/>

SELECT ?pop ?source ?confidence
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ex:Lyon ;
          rdf:predicate schema:population ;
          rdf:object ?pop ;
          ex:confidence ?confidence ;
          ex:source ?source .
}
ORDER BY DESC(?confidence)
LIMIT 1
"#;

println!();
if let QueryResults::Solutions(solutions) = store_multi.query(query_best).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let pop = num(sol.get("pop").expect("population")) as i64;
        let src = value(sol.get("source").expect("source"));
        println!("Valeur retenue (confiance max) : {} (source: {})", thousands(pop), src);
    }
};

Graphe multi-sources : 23 triplets


=== Population de Lyon selon differentes sources ===


Source                 Population  Confiance  Annee


----------------------------------------------------


INSEE                     522,250       0.95   2023


Wikipedia                 516,092       0.70   2021


Estimation mairie         530,000       0.40   2024


Valeur retenue (confiance max) : 522,250 (source: INSEE)


## Interpretation : fusion multi-sources

### Lecture des mesures

**23 triplets** :

| Part | Triplets | Calcul |
|---|---|---|
| Entite Lyon | 2 | `rdf:type`, `schema:name` |
| Trois sources | 21 | 3 × (4 structure + 3 annotations : `source`, `confidence`, `year`) |
| **Total** | **23** | |

La requete triee rend INSEE 522 250 (0,95), Wikipedia 516 092 (0,70), estimation mairie 530 000 (0,40) ; la variante `LIMIT 1` retient **522 250 (INSEE)**. Aucune valeur n'est ecrasee : le desaccord reste interrogeable, ce qui est l'interet du modele.

Deux details Rust : le separateur de milliers n'existe pas dans la bibliotheque standard, d'ou le petit helper `thousands` ; et `num()` lit la valeur d'un litteral type avant de la formater — un `f64` pour la confiance, un `i64` pour la population.

## 6. Alternative : les Graphes Nommes (Named Graphs)

Un **graphe nomme** associe un ensemble de triplets a une IRI de contexte. L'annotation porte alors sur le **contexte**, pas sur un triplet isole.

### Comparaison des approches

| Critere | Reification | Graphe nomme | Quoted triples |
|---|---|---|---|
| Granularite | un triplet | un ensemble de triplets | un triplet |
| Cout en triplets | +4 par fait | 0 (le nom est hors graphe) | 0 |
| Annoter un fait **non asserte** | oui | non (le fait doit etre dans un graphe) | oui |
| Unicite de l'annotation | par Statement | par contexte | par occurrence |
| Interoperabilite 2026 | universelle | universelle | partielle |

La question qui tranche : **l'annotation vise-t-elle un fait ou un jeu de faits ?** Pour « ces deux relations viennent du meme entretien », le graphe nomme est plus economique. Pour « cette relation precise a 0,95 de confiance », la reification (ou les quoted triples) reste necessaire.

### 6.1 Graphes nommes

`rdflib.Dataset` n'a pas d'equivalent direct dans `oxrdf` : `Graph` ne connait que le graphe par defaut. La representation idiomatique est une table ordonnee `BTreeMap<String, Graph>` — nom de contexte en cle, graphe en valeur — et un graphe par defaut separe pour les metadonnees des contextes.

```rust
let mut ds: BTreeMap<String, Graph> = BTreeMap::new();
ds.insert("alice-assertion".to_string(), g_alice);
```

Le nom du contexte joue exactement le role de l'IRI `ex:context/alice-assertion` du jumeau Python : c'est lui que le graphe par defaut annote (`ex:assertedBy`, `ex:confidence`, `ex:date`).

In [14]:
// Un graphe nomme se represente comme un couple (nom, Graph) : ici une table
// ordonnee, pour retrouver les contextes par leur nom sans dependre du hasard.
let mut ds: BTreeMap<String, Graph> = BTreeMap::new();

// Contexte "assertion d'Alice" : les triplets vivent dans le graphe nomme
let mut g_alice = Graph::new();
g_alice.insert(&Triple::new(ex("Bob"), foaf("knows"), ex("Carol")));
g_alice.insert(&Triple::new(ex("Bob"), foaf("knows"), ex("Dave")));
ds.insert("alice-assertion".to_string(), g_alice);

// Contexte "assertion de Bob"
let mut g_bob = Graph::new();
g_bob.insert(&Triple::new(ex("Carol"), foaf("knows"), ex("Eve")));
ds.insert("bob-assertion".to_string(), g_bob);

// Les annotations vivent dans le graphe par defaut
let mut g_default = Graph::new();
let ctx_alice = ex("context/alice-assertion");
g_default.insert(&Triple::new(ctx_alice.clone(), rdf("type"), ex("Assertion")));
g_default.insert(&Triple::new(ctx_alice.clone(), ex("assertedBy"), ex("Alice")));
g_default.insert(&Triple::new(ctx_alice.clone(), ex("confidence"), dbl(0.85)));
g_default.insert(&Triple::new(ctx_alice.clone(), ex("date"), date("2024-03-15")));

let ctx_bob = ex("context/bob-assertion");
g_default.insert(&Triple::new(ctx_bob.clone(), rdf("type"), ex("Assertion")));
g_default.insert(&Triple::new(ctx_bob.clone(), ex("assertedBy"), ex("Bob")));
g_default.insert(&Triple::new(ctx_bob.clone(), ex("confidence"), dbl(0.60)));

// Affichage abrege : prefixes connus, litteraux sans guillemets pour la lecture.
fn short_str(text: &str) -> String {
    let s = text.trim_start_matches('<').trim_end_matches('>');
    s.replace("http://example.org/", "ex:")
        .replace("http://www.w3.org/1999/02/22-rdf-syntax-ns#", "rdf:")
        .replace("http://xmlns.com/foaf/0.1/", "foaf:")
}
fn show_graph(g: &Graph) {
    let mut lines: Vec<(String, String, String)> = Vec::new();
    for t in g.iter() {
        let t = t.into_owned();
        let s = short_str(&t.subject.to_string());
        let p = short_str(&t.predicate.to_string());
        let o = match &t.object {
            Term::Literal(l) => l.value().to_string(),
            other => short_str(&other.to_string()),
        };
        lines.push((s, p, o));
    }
    lines.sort();
    for (s, p, o) in lines {
        println!("  {} {} {}", s, p, o);
    }
}

println!("=== Graphe par defaut (metadata) ===");
show_graph(&g_default);
println!();
println!("=== Graphe nomme : alice-assertion ===");
show_graph(ds.get("alice-assertion").expect("graphe alice"));
println!();
println!("=== Graphe nomme : bob-assertion ===");
show_graph(ds.get("bob-assertion").expect("graphe bob"));
println!();
println!("Triplets par graphe : defaut {}, alice {}, bob {}",
    g_default.len(),
    ds.get("alice-assertion").map(|g| g.len()).unwrap_or(0),
    ds.get("bob-assertion").map(|g| g.len()).unwrap_or(0));

=== Graphe par defaut (metadata) ===


  ex:context/alice-assertion ex:assertedBy ex:Alice


  ex:context/alice-assertion ex:confidence 0.85


  ex:context/alice-assertion ex:date 2024-03-15


  ex:context/alice-assertion rdf:type ex:Assertion


  ex:context/bob-assertion ex:assertedBy ex:Bob


... (4 outputs left out)

  ex:Bob foaf:knows ex:Dave


=== Graphe nomme : bob-assertion ===


  ex:Carol foaf:knows ex:Eve


Triplets par graphe : defaut 7, alice 2, bob 1


## Interpretation : graphes nommes

### Lecture des mesures

| Graphe | Triplets | Contenu |
|---|---|---|
| Par defaut (metadonnees) | 7 | 4 annotations pour `alice-assertion`, 3 pour `bob-assertion` |
| `alice-assertion` | 2 | `Bob connait Carol`, `Bob connait Dave` |
| `bob-assertion` | 1 | `Carol connait Eve` |

Le meme contenu que la section 4 aurait coute **22 triplets** en reification : trois faits reifies, soit 3 × 5 = 15 triplets d'assertion et de structure, plus les 7 annotations portees ici une seule fois par contexte. Ici, 10 triplets suffisent (7 + 2 + 1). L'economie est reelle — mais elle a un prix :

- **L'annotation est collective** : on ne peut pas dire que `Bob connait Carol` est plus fiable que `Bob connait Dave` s'ils vivent dans le meme contexte ;
- **Le fait doit exister** : un graphe nomme ne peut pas porter l'annotation d'un fait qu'il n'asserte pas. Le fait 4 de la section 4 (Alice→Dave, affirme par Bob) est **inexprimable** ainsi ;
- **Le nom du contexte est une IRI ou une chaine** : `BTreeMap` ordonne les cles, ce qui rend l'affichage deterministe (pas d'ordre d'iteration aleatoire).

## 7. Serialisation et interoperabilite

Le choix d'un mecanisme d'annotation n'est pas seulement une question de cout : il conditionne ce que les autres outils comprendront.

### Formats et support des annotations de triplets

| Format | Serialise une reification ? | Serialise des quoted triples ? | Crate du carnet |
|---|---|---|---|
| **Turtle** | oui (noeuds `[]`) | oui sous feature `rdf-12` (non activee ici) | `oxttl` |
| **N-Triples** | oui | oui sous feature `rdf-12` (non activee ici) | `oxttl` |
| **RDF/XML** | oui | oui sous feature `rdf-12` (non activee ici) | `oxrdfxml` |
| **JSON-LD** | oui | oui sous feature `rdf-12` (non activee ici) | `oxjsonld` |

C'est l'argument decisif en faveur de la reification : **elle traverse tous les formats**, y compris ceux qui n'ont jamais entendu parler de RDF 1.2. Une annotation en quoted triples ne s'echange pas encore avec la majorite des consommateurs.

### Serialisation d'un graphe avec reification

Le meme graphe — un fait, une reification, une annotation — passe dans les quatre serialiseurs. Rien ne change dans le modele : seuls les octets different.

Note sur JSON-LD : `JsonLdSerializer` ecrit du JSON **compact**, sans indentation. C'est le serialiseur de la famille oxigraph ; il n'expose pas d'option `indent`, contrairement a rdflib. Le contenu est equivalent — un consommateur JSON-LD le relit sans difference.

In [15]:
// Fait + reification + annotation : le meme graphe, quatre syntaxes.
let mut g_ser = Graph::new();
g_ser.insert(&Triple::new(ex("Alice"), foaf("knows"), ex("Bob")));
let stmt = reify(&mut g_ser, &NamedOrBlankNode::from(ex("Alice")), &foaf("knows"), &Term::from(ex("Bob")), None);
g_ser.insert(&Triple::new(stmt.clone(), ex("confidence"), dbl(0.9)));

println!("=== Turtle ===");
println!("{}", to_turtle(&g_ser).trim());
println!();
println!("=== N-Triples ===");
println!("{}", to_ntriples(&g_ser).trim());
println!();
println!("=== RDF/XML ===");
println!("{}", to_rdfxml(&g_ser).trim());
println!();
// JsonLdSerializer ecrit du JSON-LD compact (pas d'indentation) : c'est le
// serialiseur de la famille oxigraph, il n'a pas d'option `indent`.
println!("=== JSON-LD ===");
println!("{}", to_jsonld(&g_ser).trim());

=== Turtle ===


@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .


@prefix foaf: <http://xmlns.com/foaf/0.1/> .


@prefix ex: <http://example.org/> .


ex:Alice foaf:knows ex:Bob .


... (24 outputs left out)

	</rdf:Description>


</rdf:RDF>


=== JSON-LD ===


{"@context":{"ex":"http://example.org/","foaf":"http://xmlns.com/foaf/0.1/"},"@graph":[{"@id":"http://example.org/Alice","http://xmlns.com/foaf/0.1/knows":[{"@id":"http://example.org/Bob"}]},{"@id":"_:a48ae20445b12de00a3ae60be62ce983","http://www.w3.org/1999/02/22-rdf-syntax-ns#object":[{"@id":"http://example.org/Bob"}],"http://www.w3.org/1999/02/22-rdf-syntax-ns#type":[{"@id":"http://www.w3.org/1999/02/22-rdf-syntax-ns#Statement"}],"http://www.w3.org/1999/02/22-rdf-syntax-ns#subject":[{"@id":"http://example.org/Alice"}],"http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate":[{"@id":"http://xmlns.com/foaf/0.1/knows"}],"http://example.org/confidence":[{"@type":"http://www.w3.org/2001/XMLSchema#double","@value":"0.9"}]}]}


## Interpretation : formats de serialisation

### Lecture des mesures

| Format | Forme produite | Remarque de lecture |
|---|---|---|
| Turtle | deux sujets : le fait, puis le Statement | prefixes declares ; le noeud blanc est ecrit en entier, `oxrdf` ne compacte pas en `[]` |
| N-Triples | une ligne par triplet | etiquette de noeud blanc en hexadecimal (`_:a48ae204…` dans la sortie ci-dessus), regeneree a chaque execution |
| RDF/XML | un `rdf:Description` (Alice) et un `rdf:Statement` (le noeud blanc) | le noeud blanc porte `rdf:nodeID` |
| JSON-LD | un objet `@graph` a deux entrees | compact, une entree par sujet |

Le meme graphe de 6 triplets, quatre ecritures. Trois consequences pratiques :

- **N-Triples est canonique par ligne** : trier les lignes donne une forme stable, sauf pour les etiquettes de noeuds blancs — d'ou l'interet des IRI nommees quand une comparaison octet a octet compte ;
- **Turtle est le format de lecture humaine** : le regroupement par sujet — le fait d'un cote, le Statement de l'autre — rend la structure visible ;
- **JSON-LD est le format d'echange web** : c'est celui qu'un service consomme sans savoir ce qu'est RDF.

## 8. Comparaison des trois approches

| Critere | Reification | Graphes nommes | Quoted triples (RDF 1.2) |
|---|---|---|---|
| Triplets pour 4 faits annotes (3 annotations) | 4 × 8 = 32 | ~12 + 4 | 8 |
| Annoter un fait non asserte | oui | non | oui |
| Annotation par triplet | oui | non | oui |
| Format d'echange universel | oui | oui | non (2026) |
| Complexite des requetes | jointure sur 4 predicats | jointure sur le graphe | patterns imbriques |
| Outillage Rust disponible | complet | complet (table de graphes) | modele et serialiseurs derriere `rdf-12` (non activee ici) |

### Recommandations

1. **Interoperabilite d'abord** : reification. C'est le seul mecanisme que tous les consommateurs comprennent aujourd'hui, et le seul que `oxttl` 0.2 / `oxjsonld` 0.2 serialisent **sans feature** ;
2. **Annotation collective** (une source pour un lot de faits) : graphes nommes, plus economiques ;
3. **Volume et expressivite** : quoted triples. Cote Rust, la pile est prete — `oxrdf` porte `Term::Triple`, et `oxttl` / `oxrdfxml` / `oxjsonld` declarent `rdf-12` ; il ne manque qu'un carnet qui active la feature et en assume le modele a quatre variantes ;
4. **Ne pas melanger sans raison** : un graphe qui utilise les trois mecanismes pour le meme besoin est illisible. Choisir, documenter, et garder la porte ouverte par une migration scriptee.

## 9. Exercices et exemples guides

Les exemples guides 1 a 3 sont resolus et executes ; les exercices 4 a 6 sont a completer. Tous suivent la meme trame : construire un graphe annote, ecrire la requete qui repond a la question metier, verifier que le resultat correspond a l'intuition.

### Exemple guide 1 : critiques de film avec reification

Un film recoit trois notes de trois critiques. Le fait `ex:Film1 ex:rating 4` et le fait `ex:Film1 ex:rating 5` sont **deux triplets differents** : c'est ce qui rend la reification necessaire — sans elle, on ne saurait pas quelle note vient de quel critique.

La requete trie par note decroissante et rend, pour chaque critique, sa note, son media et sa date.

In [16]:
// Exemple guide 1 : critiques de film avec reification
let mut g_ex1 = Graph::new();

// Film
g_ex1.insert(&Triple::new(ex("Film1"), rdf("type"), ex("Film")));
g_ex1.insert(&Triple::new(ex("Film1"), ex("title"), txt("Les Temps Modernes")));

// Les trois notes sont des triplets distincts, donc ont chacune leur reification.
for (note, reviewer, source, jour) in [
    (4, "Critique A", "Le Monde", "2024-01-15"),
    (3, "Critique B", "Telerama", "2024-01-20"),
    (5, "Critique C", "Cahiers du Cinema", "2024-02-01"),
] {
    let value = int(note);
    g_ex1.insert(&Triple::new(ex("Film1"), ex("rating"), value.clone()));
    let stmt = reify(&mut g_ex1, &NamedOrBlankNode::from(ex("Film1")), &ex("rating"), &Term::from(value), None);
    g_ex1.insert(&Triple::new(stmt.clone(), ex("reviewer"), txt(reviewer)));
    g_ex1.insert(&Triple::new(stmt.clone(), ex("source"), txt(source)));
    g_ex1.insert(&Triple::new(stmt.clone(), ex("date"), date(jour)));
}

println!("=== Graphe des critiques ===");
println!("{}", to_turtle(&g_ex1));

// Requete SPARQL : lister les critiques
let query_notes = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
SELECT ?note ?reviewer ?source ?date
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ex:Film1 ;
          rdf:predicate ex:rating ;
          rdf:object ?note ;
          ex:reviewer ?reviewer ;
          ex:source ?source ;
          ex:date ?date .
}
ORDER BY DESC(?note)
"#;

println!();
println!("=== Critiques triees par note ===");
let store_ex1 = store_of(&g_ex1);
if let QueryResults::Solutions(solutions) = store_ex1.query(query_notes).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let note = value(sol.get("note").expect("note"));
        let reviewer = value(sol.get("reviewer").expect("critique"));
        let source = value(sol.get("source").expect("source"));
        let jour = value(sol.get("date").expect("date"));
        println!("  {} : {}/5 ({}, {})", reviewer, note, source, jour);
    }
};

=== Graphe des critiques ===


@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .


@prefix foaf: <http://xmlns.com/foaf/0.1/> .


@prefix ex: <http://example.org/> .


ex:Film1 ex:title "Les Temps Modernes" ;


... (23 outputs left out)

=== Critiques triees par note ===


  Critique C : 5/5 (Cahiers du Cinema, 2024-02-01)


  Critique A : 4/5 (Le Monde, 2024-01-15)


  Critique B : 3/5 (Telerama, 2024-01-20)


### Exemple guide 2 : requeter les reifications avec SPARQL

Deux questions sur le graphe de la section 4.1 :

- **2a** — quelles relations sont affirmees par un tiers ? (jointure sur `ex:assertedBy`, le fait peut ne pas etre asserte)
- **2b** — quelles relations datent d'avant 2022 ? (filtre sur un litteral `xsd:date`, comparaison de dates, pas de chaines)

La seconde montre un piege classique : `FILTER(?since < "2022-01-01"^^xsd:date)` compare des dates ; sans le type, la comparaison porterait sur la forme textuelle et donnerait un autre resultat sur d'autres jeux.

In [17]:
// Exemple guide 2 : requetes SPARQL sur les reifications du graphe g_kg

// Requete 2a : relations assertees par une tierce personne
let query_2a = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?person1 ?person2 ?asserter
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?person1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?person2 ;
          ex:assertedBy ?asserter .
}
"#;

// Requete 2b : relations anciennes (avant 2022)
let query_2b = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?person1 ?person2 ?since
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?person1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?person2 ;
          ex:since ?since .
    FILTER(?since < "2022-01-01"^^xsd:date)
}
"#;

println!("=== 2a : Relations assertees par un tiers ===");
if let QueryResults::Solutions(solutions) = store_kg.query(query_2a).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let p1 = local(sol.get("person1").expect("p1"));
        let p2 = local(sol.get("person2").expect("p2"));
        let who = local(sol.get("asserter").expect("assertant"));
        println!("  {} affirme : {} connait {}", who, p1, p2);
    }
}

println!();
println!("=== 2b : Relations anterieures a 2022 ===");
if let QueryResults::Solutions(solutions) = store_kg.query(query_2b).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let p1 = local(sol.get("person1").expect("p1"));
        let p2 = local(sol.get("person2").expect("p2"));
        let since = value(sol.get("since").expect("depuis"));
        println!("  {} connait {} depuis {}", p1, p2, since);
    }
};

=== 2a : Relations assertees par un tiers ===


  Bob affirme : Alice connait Dave


=== 2b : Relations anterieures a 2022 ===


  Bob connait Carol depuis 2021-07-15


  Alice connait Bob depuis 2019-03-01


### Exemple guide 3 : graphe de confiance avec RDF-Star

Cinq capitales, cinq sources, cinq confiances. La requete ne garde que les affirmations a 0,8 ou plus — la Suisse et sa source « a verifier » (0,75) sortent du resultat, alors que le fait `ex:Suisse ex:capital ex:Berne` reste dans le graphe.

C'est la difference entre **filtrer une annotation** et **supprimer un fait** : la reification permet le premier sans le second.

In [18]:
// Exemple guide 3 : graphe de confiance avec RDF-Star
let mut g_trust = Graph::new();

let capital_facts = [
    ("France", "Paris", 0.98, "Wikidata", "2024-03-01"),
    ("Allemagne", "Berlin", 0.97, "DBpedia", "2024-03-02"),
    ("Italie", "Rome", 0.96, "Cours geographie", "2024-03-03"),
    ("Espagne", "Madrid", 0.95, "Wikidata", "2024-03-04"),
    ("Suisse", "Berne", 0.75, "Source a verifier", "2024-03-05"),
];

for (country, capital, confidence, source, timestamp) in capital_facts {
    g_trust.insert(&Triple::new(ex(country), ex("capital"), ex(capital)));
    let stmt = reify(&mut g_trust, &NamedOrBlankNode::from(ex(country)), &ex("capital"), &Term::from(ex(capital)), None);
    g_trust.insert(&Triple::new(stmt.clone(), ex("confidence"), dbl(confidence)));
    g_trust.insert(&Triple::new(stmt.clone(), ex("source"), txt(source)));
    g_trust.insert(&Triple::new(stmt.clone(), ex("timestamp"), date(timestamp)));
}

println!("Graphe de confiance : {} triplets\n", g_trust.len());
println!("{}", to_turtle(&g_trust));

let query_trusted = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?country ?capital ?confidence ?source ?timestamp
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?country ;
          rdf:predicate ex:capital ;
          rdf:object ?capital ;
          ex:confidence ?confidence ;
          ex:source ?source ;
          ex:timestamp ?timestamp .
    FILTER (?confidence >= "0.8"^^xsd:double)
}
ORDER BY DESC(?confidence)
"#;

println!("=== Capitales avec confiance >= 0.8 ===");
let store_trust = store_of(&g_trust);
if let QueryResults::Solutions(solutions) = store_trust.query(query_trusted).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let country = local(sol.get("country").expect("pays"));
        let capital = local(sol.get("capital").expect("capitale"));
        let conf = num(sol.get("confidence").expect("confiance"));
        let src = value(sol.get("source").expect("source"));
        println!("  {} -> {} (confiance={:.2}, source={})", country, capital, conf, src);
    }
};

Graphe de confiance : 40 triplets


@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .


@prefix foaf: <http://xmlns.com/foaf/0.1/> .


@prefix ex: <http://example.org/> .


ex:Suisse ex:capital ex:Berne .


... (40 outputs left out)

  France -> Paris (confiance=0.98, source=Wikidata)


  Allemagne -> Berlin (confiance=0.97, source=DBpedia)


  Italie -> Rome (confiance=0.96, source=Cours geographie)


  Espagne -> Madrid (confiance=0.95, source=Wikidata)


## Exemples guides (solutions proposees par @Sosolalt)

*Les exemples ci-dessous ont ete resolus par @Sosolalt (EPITA-IS, promo 2028) dans le jumeau Python.*
*Ils sont reproduits ici avec les memes donnees, pour que les mesures soient comparables d'un carnet a l'autre.*

### Exemple guide 4 : donnees sportives multi-sources

Six affirmations de score reparties sur trois matchs. La question est celle d'un lecteur de resultats : **quel score est confirme ?** La requete agrege par `(match, score)` et ne garde que les groupes ou `COUNT(DISTINCT ?source) >= 2`.

Deux pieges y sont traites : `COUNT(DISTINCT …)` compte les sources, pas les triplets — une meme source qui repete le score ne confirme rien ; et le `HAVING` porte sur l'agregat, pas sur une variable liee.

In [19]:
// Exemple guide 4 : donnees sportives multi-sources
let mut g_sport = Graph::new();

// Helper : declare un match (equipes) et un score affirme par une source.
fn add_score_claim(
    g: &mut Graph,
    match_id: &str,
    home: &str,
    away: &str,
    score: &str,
    source: &str,
    confidence: f64,
    jour: &str,
) {
    g.insert(&Triple::new(ex(match_id), ex("homeTeam"), txt(home)));
    g.insert(&Triple::new(ex(match_id), ex("awayTeam"), txt(away)));
    // Le score est l'assertion annotee par source -> reification
    let stmt = reify(g, &NamedOrBlankNode::from(ex(match_id)), &ex("score"), &Term::from(txt(score)), None);
    g.insert(&Triple::new(stmt.clone(), ex("source"), txt(source)));
    g.insert(&Triple::new(stmt.clone(), ex("confidence"), dbl(confidence)));
    g.insert(&Triple::new(stmt.clone(), ex("date"), date(jour)));
}

// Match 1 : confirme par 2 sources (meme score 3-1)
add_score_claim(&mut g_sport, "match1", "PSG", "OM", "3-1", "L'Equipe", 0.90, "2024-03-10");
add_score_claim(&mut g_sport, "match1", "PSG", "OM", "3-1", "Le Monde", 0.85, "2024-03-10");

// Match 2 : une seule source -> non confirme
add_score_claim(&mut g_sport, "match2", "Lyon", "Nice", "2-2", "L'Equipe", 0.80, "2024-03-11");

// Match 3 : confirme par 3 sources (meme score 0-1)
add_score_claim(&mut g_sport, "match3", "Lille", "Lens", "0-1", "L'Equipe", 0.70, "2024-03-12");
add_score_claim(&mut g_sport, "match3", "Lille", "Lens", "0-1", "Le Monde", 0.75, "2024-03-12");
add_score_claim(&mut g_sport, "match3", "Lille", "Lens", "0-1", "Foot Mercato", 0.60, "2024-03-12");

println!("Graphe sportif : {} triplets\n", g_sport.len());

// Requete : scores confirmes par au moins 2 sources differentes
let query_confirmed = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>

SELECT ?match ?score (COUNT(DISTINCT ?source) AS ?nbSources)
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?match ;
          rdf:predicate ex:score ;
          rdf:object ?score ;
          ex:source ?source .
}
GROUP BY ?match ?score
HAVING (COUNT(DISTINCT ?source) >= 2)
ORDER BY ?match
"#;

println!("=== Scores confirmes par >= 2 sources ===");
let store_sport = store_of(&g_sport);
if let QueryResults::Solutions(solutions) = store_sport.query(query_confirmed).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let match_id = local(sol.get("match").expect("match"));
        let score = value(sol.get("score").expect("score"));
        let nb = value(sol.get("nbSources").expect("sources"));
        println!("  {} : score {} (confirme par {} sources)", match_id, score, nb);
    }
};

Graphe sportif : 48 triplets


=== Scores confirmes par >= 2 sources ===


  match1 : score 3-1 (confirme par 2 sources)


  match3 : score 0-1 (confirme par 3 sources)


### Exemple guide 5 : agregats SPARQL sur scores de confiance

Trois agregats sur le graphe de la section 4.1 :

1. **`AVG`** — la confiance moyenne des relations `foaf:knows` ;
2. **`ORDER BY ASC` + `LIMIT 1`** — la relation la plus ancienne ;
3. **`GROUP BY ?source`** — le nombre de relations par source.

La moyenne porte sur des `xsd:double` : `AVG` rend un double, pas une chaine. C'est le point ou un litteral non type casserait la requete.

In [20]:
// Exemple guide 5 : agregats SPARQL sur scores de confiance (graphe g_kg, section 4)

// 1. Moyenne des scores de confiance de toutes les relations foaf:knows
let query_avg = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT (AVG(?conf) AS ?moyenne) (COUNT(?stmt) AS ?nbRelations)
WHERE {
    ?stmt a rdf:Statement ;
          rdf:predicate foaf:knows ;
          ex:confidence ?conf .
}
"#;

println!("=== 1. Moyenne des scores de confiance (foaf:knows) ===");
if let QueryResults::Solutions(solutions) = store_kg.query(query_avg).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let moyenne = num(sol.get("moyenne").expect("moyenne"));
        let nb = num(sol.get("nbRelations").expect("compte")) as i64;
        // (0.95 + 0.60 + 0.30 + 0.50) / 4 = 0.5875 : le resultat exact tombe au milieu
        // de deux arrondis a trois decimales. Selon l'ordre de la somme flottante, le
        // double vaut 0.58749999... ou 0.5875, et l'affichage a trois decimales bascule
        // entre 0.587 et 0.588 d'une execution a l'autre (ordre de parcours du magasin).
        // Quatre decimales affichent le nombre exact, identique a chaque execution.
        println!("  Moyenne = {:.4} sur {} relations", moyenne, nb);
    }
}

// 2. Relation la plus ancienne (date ex:since minimale)
let query_oldest = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?p1 ?p2 ?since
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?p1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?p2 ;
          ex:since ?since .
}
ORDER BY ASC(?since)
LIMIT 1
"#;

println!();
println!("=== 2. Relation la plus ancienne ===");
if let QueryResults::Solutions(solutions) = store_kg.query(query_oldest).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let p1 = local(sol.get("p1").expect("p1"));
        let p2 = local(sol.get("p2").expect("p2"));
        let since = value(sol.get("since").expect("depuis"));
        println!("  {} connait {} depuis {}", p1, p2, since);
    }
}

// 3. Nombre de relations par source
let query_by_source = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?source (COUNT(?stmt) AS ?nb)
WHERE {
    ?stmt a rdf:Statement ;
          rdf:predicate foaf:knows ;
          ex:source ?source .
}
GROUP BY ?source
ORDER BY DESC(?nb)
"#;

println!();
println!("=== 3. Nombre de relations par source ===");
if let QueryResults::Solutions(solutions) = store_kg.query(query_by_source).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let source = value(sol.get("source").expect("source"));
        let nb = num(sol.get("nb").expect("compte")) as i64;
        println!("  {:<15} : {} relation(s)", source, nb);
    }
};

=== 1. Moyenne des scores de confiance (foaf:knows) ===


  Moyenne = 0.5875 sur 4 relations


=== 2. Relation la plus ancienne ===


  Alice connait Bob depuis 2019-03-01


=== 3. Nombre de relations par source ===


  Rumeur          : 1 relation(s)


  LinkedIn        : 1 relation(s)


  Twitter         : 1 relation(s)


### Exemple guide 6 : annotations temporelles sur des evenements historiques

Trois evenements, chacun avec sa date assertee et sa source annotee. La requete filtre sur `YEAR(?date) > 1800` : la prise de la Bastille (1789) sort, le sacre de Napoleon (1804) et la chute du mur (1989) restent.

`YEAR()` est une fonction SPARQL 1.1 standard ; elle exige un litteral `xsd:date`, ce qui rend le typage des dates obligatoire — en Rust comme en Python, mais visible different : `date("1804-12-02")` nomme le type a la construction.

In [21]:
// Exemple guide 6 : annotations temporelles sur des evenements historiques
// Le fait date est asserte, puis annote (date + source) par reification :
// c'est la forme standard equivalente a << ?evenement ex:date ?date >> ex:source ?source .
let mut g_events = Graph::new();

let evenements = [
    ("priseBastille", "Prise de la Bastille", "1789-07-14", "Encyclopedie Larousse"),
    ("sacreNapoleon", "Sacre de Napoleon Ier", "1804-12-02", "Archives nationales"),
    ("chuteMurBerlin", "Chute du mur de Berlin", "1989-11-09", "Wikipedia"),
];

for (uri, label, jour, source) in evenements {
    g_events.insert(&Triple::new(ex(uri), rdf("type"), ex("EvenementHistorique")));
    g_events.insert(&Triple::new(ex(uri), ex("label"), txt(label)));
    let jour_value = date(jour);
    g_events.insert(&Triple::new(ex(uri), ex("date"), jour_value.clone()));
    let stmt = reify(&mut g_events, &NamedOrBlankNode::from(ex(uri)), &ex("date"), &Term::from(jour_value), None);
    g_events.insert(&Triple::new(stmt.clone(), ex("source"), txt(source)));
}

println!("Graphe d'evenements : {} triplets\n", g_events.len());

// Requete SPARQL filtrant les evenements posterieurs a l'an 1800
let query_events = r#"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex: <http://example.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?label ?date ?source
WHERE {
    ?event ex:label ?label ;
           ex:date ?date .
    ?stmt a rdf:Statement ;
          rdf:subject ?event ;
          rdf:predicate ex:date ;
          rdf:object ?date ;
          ex:source ?source .
    FILTER (YEAR(?date) > 1800)
}
ORDER BY ?date
"#;

println!("=== Evenements posterieurs a 1800 ===");
let store_events = store_of(&g_events);
if let QueryResults::Solutions(solutions) = store_events.query(query_events).expect("requete") {
    for solution in solutions {
        let sol = solution.expect("solution");
        let label = value(sol.get("label").expect("libelle"));
        let jour = value(sol.get("date").expect("date"));
        let source = value(sol.get("source").expect("source"));
        println!("  {} - {} (source : {})", jour, label, source);
    }
};

Graphe d'evenements : 24 triplets


=== Evenements posterieurs a 1800 ===


  1804-12-02 - Sacre de Napoleon Ier (source : Archives nationales)


  1989-11-09 - Chute du mur de Berlin (source : Wikipedia)


## Exercices

## Exercices a completer

Les trois exercices suivants reprennent les exemples guides en retirant la solution. Le carnet doit s'executer de bout en bout : chaque cellule d'exercice affiche `Exercice a completer` tant que le travail n'est pas fait.

### Exercice 4 : Annoter des donnees sportives multi-sources

Construisez un graphe de matchs : equipes, scores, et pour chaque score la source qui l'affirme. Ecrivez ensuite la requete qui ne retient que les scores confirmes par au moins deux sources distinctes.

Indications : `reify(&mut g, …)` pour chaque affirmation, `store_of(&g)` pour obtenir le `Store` interrogeable, `QueryResults::Solutions` pour parcourir les resultats.

In [22]:
// Exercice 4 : donnees sportives multi-sources
// TODO etudiant : construisez le graphe (matchs, scores, sources multiples) avec
//   reify(), puis ecrivez une requete SPARQL qui trouve les scores confirmes
//   par au moins 2 sources. Rappel de la boucle : store_of(&graphe), puis
//   store.query(...) -> QueryResults::Solutions (cf. section 4.2).

println!("Exercice a completer");

Exercice a completer


### Exercice 5 : Agregats SPARQL sur scores de confiance

Sur le graphe `g_kg` construit en section 4.1, ecrivez trois requetes :

1. la moyenne des confiances (`AVG`) ;
2. la relation la plus ancienne (`ORDER BY ASC(?since)` + `LIMIT 1`) ;
3. le nombre de relations par source (`GROUP BY ?source`).

Verifiez vos resultats contre ceux de l'exemple guide 5 : ils doivent coincider.

In [23]:
// Exercice 5 : agregats SPARQL sur scores de confiance
// TODO etudiant : ecrivez 3 requetes SPARQL avec agregats sur le graphe g_kg
//   1. moyenne des confiances (AVG)
//   2. relation la plus ancienne (ORDER BY ASC + LIMIT 1)
//   3. nombre de relations par source (GROUP BY)
//   Affichez chaque resultat : les solutions arrivent dans QueryResults::Solutions.

println!("Exercice a completer");

Exercice a completer


### Exercice 6 : Annotations temporelles sur des evenements historiques

#### Objectifs

Construire un graphe d'evenements historiques, annoter chaque date par sa source, et filtrer par annee.

1. Creer un `Graph` et ajouter trois evenements avec leur date assertee ;
2. Reifier chaque fait date pour y attacher la source ;
3. Ecrire la requete qui ne garde que les evenements posterieurs a 1800 ;
4. Afficher les resultats.

Le carnet fournit deja les helpers (`ex`, `rdf`, `date`, `txt`, `reify`, `store_of`) : l'exercice porte sur la construction et la requete, pas sur la plomberie.

In [24]:
// Exercice 6 : annotations temporelles sur des evenements historiques
// TODO etudiant : creez un graphe d'evenements historiques annotes
//
// Etape 1 : le graphe et les helpers sont deja charges (ex(), rdf(), date(), txt())
// Etape 2 : creez un Graph, ajoutez 3 evenements avec leur date assertee, puis
//   reifiez chaque fait date pour y attacher sa source
// Etape 3 : ecrivez une requete SPARQL filtrant les evenements posterieurs a 1800
//   (FILTER (YEAR(?date) > 1800))
// Etape 4 : affichez les resultats

let mut g = Graph::new();
let mut results: Vec<String> = Vec::new();
let _ = &mut g;
let _ = &mut results;
println!("Exercice a completer");

Exercice a completer


## Resume

### Concepts cles

- **Reification** : un fait devient une ressource (`rdf:Statement`) decrite par 4 triplets ; les annotations se posent sur cette ressource. Universelle, couteuse ;
- **Graphes nommes** : l'annotation porte sur un ensemble de triplets reunis par une IRI de contexte. Economique, non granulaire ;
- **Quoted triples (RDF 1.2)** : un triplet peut occuper une position de sujet ou d'objet. Le plus expressif, le moins interoperable aujourd'hui — `oxrdf` le porte derriere la feature `rdf-12` ;
- **Un fait non asserte est representable** en reification : seule sa decomposition existe. C'est l'usage central (croyances, historique, desaccords).

### Competences acquises

- Construire et interroger un graphe annote avec `oxrdf` et `oxigraph` ;
- Lire la difference entre filtrer une annotation et supprimer un fait ;
- Choisir un mecanisme d'annotation sur des criteres mesures (triplets, granularite, interoperabilite) ;
- Serialiser un meme graphe dans quatre formats et savoir ce que chacun preserve.

## Annexe A - Ecosysteme et compatibilite (lecture optionnelle)

### Support dans les triple stores

| Magasin | Reification | Graphes nommes | Quoted triples |
|---|---|---|---|
| Apache Jena / Fuseki | oui | oui | partiel (TDB2) |
| GraphDB | oui | oui | non |
| Blazegraph | oui | oui | non |
| oxigraph | oui | oui | via RDF 1.2 en amont |
| Stardog | oui | oui | oui |

### Support dans les bibliotheques

| Langage | Bibliotheque | Reification | Quoted triples |
|---|---|---|---|
| Python | rdflib 7.6 | oui | non (le jumeau Python le documente) |
| C# | dotNetRDF | oui | non (jumeau C#) |
| Rust | oxrdf 0.3.5 / oxigraph 0.5 | oui | modele et serialiseurs derriere `rdf-12` (feature declaree par oxttl, oxrdfxml et oxjsonld ; non activee ici) |

Cette annexe n'est pas a retenir : elle sert a situer le choix quand un projet doit s'echanger avec un systeme precis. Le critere qui prime reste celui de la section 8 — ce que le consommateur comprend.

## Conclusion

RDF-Star repond a un besoin aussi vieux que RDF : **dire quelque chose sur un fait**. La reponse historique — la reification — est verbeuse mais universelle ; la reponse moderne — les quoted triples — est concise mais encore peu serialisee. Entre les deux, les graphes nommes couvrent l'annotation collective.

Ce carnet a mesure les trois sur le meme contenu : 37 triplets pour un graphe de connaissances annote en reification, 10 pour le meme genre de contenu en graphes nommes, 12 pour une imbrication a deux niveaux qui tiendrait en 3 triplets en RDF 1.2 (le fait de base et ses deux annotations). Aucun de ces nombres n'est un argument suffisant seul : le choix se fait sur le triplet (granularite, non-assertion, interoperabilite).

Cote Rust, la pile `oxrdf` + `oxigraph` fournit tout ce que le carnet demande, y compris les quatre serialiseurs, avec une difference de fond assumee : le typage est explicite et les erreurs sont des `Result`. C'est plus verbeux a l'ecriture, et plus sur a l'execution — le compilateur refuse une IRI invalide oubliee, la ou Python la decouvre a l'execution.

***

Jumeau Rust du carnet [SW-10-RDFStar-Python](SW-10-RDFStar-Python.ipynb). Exemples guides 4 a 6 : solutions proposees par @Sosolalt dans le jumeau Python. Pile : famille oxigraph (`MIT OR Apache-2.0`), executee sur le noyau `rust` (evcxr). Mesures de conformite : `scripts/notebook_tools/rust_conformance.d/sw-10-rdfstar.yaml`.